# 03 — Paged KV cache (Phase 3)

Paging is a **capacity** optimization. Expect decode to get slightly
slower (the gather is a real copy of the live KV, per layer, per step)
and expect far more concurrency to fit under a fixed VRAM budget.

Experiment A needs no GPU and runs in seconds — it is arithmetic about
the allocation policy, not about the T4. Only Experiment B needs the
card.

**Budget:** ~2 minutes for A, ~30 minutes for B.

In [ ]:
import os

# Set before torch is imported anywhere, and before any `!python` child
# process inherits the environment.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Two ways in. If you attached the repo as a Kaggle Dataset, point
# UPLOAD at it; otherwise it clones. A private repo needs a PAT in
# Add-ons -> Secrets as GITHUB_TOKEN.
REPO  = "https://github.com/Jash-Vora/LatentServe.git"
DEST  = Path("/kaggle/working/LatentServe")
UPLOAD = Path("/kaggle/input/latentserve")   # optional dataset fallback

if not DEST.exists():
    if UPLOAD.exists():
        !cp -r {UPLOAD} {DEST}
    else:
        token = ""
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception:
            pass  # public repo, or you're pulling a fork
        url = REPO.replace("https://", f"https://{token}@") if token else REPO
        subprocess.run(["git", "clone", "--depth", "1", url, str(DEST)], check=True)

os.chdir(DEST)
sys.path.insert(0, str(DEST))
os.environ["PYTHONPATH"] = f"{DEST}:" + os.environ.get("PYTHONPATH", "")
print("cwd:", Path.cwd())
!git log --oneline -1

In [ ]:
# Do NOT install torch here. Kaggle ships a build matched to its driver;
# replacing it is the fastest way to lose an afternoon.
!pip install -q pydantic pyyaml 2>&1 | tail -1

import torch, transformers
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| device", torch.cuda.get_device_name(0))
print("transformers", transformers.__version__)

major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
if (major, minor) < (4, 40):
    print("\n[!] pyproject requires transformers>=4.40. Run:  !pip install -q -U transformers")
else:
    print("transformers OK (LatentServe's own layer loop is version-independent "
          "by design, but model/qwen.py's HF cache walk needs >=4.40)")

In [ ]:
# Pull the checkpoint once, up front, so the download (~3 GB) never lands
# inside a timed benchmark. Later sessions re-download unless you save the
# HF cache as a Kaggle Dataset — worth doing once you're iterating daily.
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

cfg = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={cfg.num_hidden_layers} q_heads={cfg.num_attention_heads} "
      f"kv_heads={cfg.num_key_value_heads} "
      f"group={cfg.num_attention_heads // cfg.num_key_value_heads} "
      f"head_dim={getattr(cfg, 'head_dim', None) or cfg.hidden_size // cfg.num_attention_heads}")
_ = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
_ = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype="float16")
del _
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("cached")

## 1. Gate 4 precondition — paging must not change the arithmetic

Contiguous vs. paged logits at block sizes 1, 4 and 16, plus allocator
behaviour under churn and refcounting. Block size 1 is the maximally
scattered case, where any slot-mapping error shows up immediately.

In [ ]:
!python -m pytest tests/test_phase3_paged.py -v 2>&1 | tail -30

## 2. Experiment A — capacity (CPU, ~2 min)

Two contiguous baselines: `generic` is sized for the longest request the
server accepts, `oracle` is sized to the longest sequence this workload
actually produces. The oracle cannot be built in practice — it needs the
future — and is here so paging has to beat the best possible contiguous
configuration rather than a strawman.

In [ ]:
!python -m benchmarks.runners.phase3_paged --experiment capacity \
    --workloads mixed long_generation long_prompt short_interactive \
    --block-sizes 1 8 16 32 64 128 256 \
    --budget-gb 8 --num-requests 200

## 3. Experiment B — what paging costs (T4, ~30 min)

Same points as Phase 2's Run B so the numbers are directly comparable.
Two block sizes, because the gather's cost depends on how scattered the
blocks are.

In [ ]:
!python -m benchmarks.runners.phase3_paged --experiment latency \
    --config configs/phase3_paged.yaml \
    --context-lengths 4096 8192 16384 --batch-sizes 1 4 \
    --latency-block-sizes 16 128 \
    --output-tokens 128 --repeats 3 --warmup 1

## 4. Read it back

In [ ]:
import pandas as pd

cap = pd.read_json("results/raw/phase3_capacity.jsonl", lines=True)
cap = pd.concat([cap.drop(columns=["extra"]), pd.json_normalize(cap["extra"])], axis=1)
print(cap.pivot_table(index=["workload", "policy", "block_size"],
                      values=["tokens_per_gb", "capacity_efficiency",
                              "mean_concurrency", "alloc_overhead_us_per_token"],
                      dropna=False).round(3).to_string())

In [ ]:
lat = pd.read_json("results/raw/phase3_paged.jsonl", lines=True)
lat = pd.concat([lat.drop(columns=["extra"]), pd.json_normalize(lat["extra"])], axis=1)
ok = lat[lat.status == "ok"]
print(ok[["policy", "block_size", "batch_size", "context_length",
          "tpot_ms", "ttft_ms", "paging_overhead_pct", "gather_mb_per_step"]]
      .sort_values(["batch_size", "context_length", "policy"]).to_string(index=False))
print("\nIf any paged row is FASTER than its contiguous row, that is a "
      "measurement error, not a win — the gather cannot be free.")

## 5. Before you close the session

Fill in the Gate 4 checklist in `docs/phase3.md`, state paging's
advantage against the **oracle** baseline rather than only the generic
one, and pick a block size with the fragmentation and overhead numbers
that justify it. Then zip the results — `results/raw/*.jsonl` is
gitignored and exists only as notebook output.

In [ ]:
!cd /kaggle/working/LatentServe && zip -qr /kaggle/working/results_phase3.zip results/raw
!ls -la /kaggle/working/*.zip